In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm=ChatOpenAI(model="gpt-5-nano",temperature=0)
response=llm.invoke("What is AI ? Tell me in one line")


In [ ]:
response.content

### RAG IMPLEMENTATION WITH YOU OWN TEXT DATA

#### step 1 : Preparing document for your text

In [ ]:
from langchain_core.documents import Document

my_text="""Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]

High-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.

The traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics.[a] To reach these goals, AI researchers have used techniques including state space search and mathematical optimization, formal logic, artificial neural networks, and methods based on statistics, operations research, and economics.[b] AI also draws upon psychology, linguistics, philosophy, neuroscience, and other fields.[2] Some companies, such as OpenAI, Google DeepMind and Meta, aim to create artificial general intelligence (AGI) – AI that can complete virtually any cognitive task at least as well as a human.[3]"""


docs = [
    Document(
        page_content=my_text,
        metadata={
            "source": "vk_org",
            "document_id": "doc-1"
        }
    )
]
docs

In [ ]:
### Step 2 - Splitting the dcument in to chunks

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50)

chunks=splitter.split_documents(docs)
chunks

In [ ]:
###STEP 3 - Creating Embedding for the chunks


In [ ]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

In [ ]:
##example  - The output is the vector created for the text
embedding_model.embed_query("What is AI?")

### STEP 4 - Create and Store Embedding / Vectors to Vector Store

In [ ]:
from langchain_community.vectorstores import Chroma

vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model
)

In [ ]:
vectorstore.similarity_search("What is AI?",k=3)

In [ ]:
###STEP 5 SEMANTIC SEARCH 
# Talk to llm

In [ ]:
context = vectorstore.similarity_search("What is AI?",k=3)

In [ ]:
context

In [ ]:
response=llm.invoke(f"What is AI? you can answer using the following context :{context}")
print(response.content)